DATA PREPROSESSING : 

In [1]:
#import required libraries
import re  # re - regrex
import pandas as pd


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Pratham\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Pratham\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Pratham\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Pratham\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Pratham\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Pratham\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



In [ ]:
f = open('PUT_UR_OWN_WHATSAPP_CHAT.txt','r', encoding= 'utf8')   # file handling only read 

In [ ]:
data = f.read()

In [ ]:
print(data)

In [ ]:
# We have to separarte date, time, message so it will we suitable for pandas to handle the data 

In [ ]:
# to verify pattern between this 21/11/23, 4:16 pm - Pratham Harer: <Media omitted> 
# use ---> search Regrx101 on google 

In [ ]:
pattern= r'\d{1,2}\/\d{1,2}\/\d{2},\s\d{1,2}:\d{2}\s(?:am|pm)\s-\s'   # generate pattern from chatgpt and verify using regrex101

In [ ]:
messages = re.split(pattern, data)[1:]  # why [1:] ?  coz, In list first element is empty to avoid that
messages

In [ ]:
dates = re.findall(pattern, data)
dates = [d.replace('\u202f', ' ') for d in dates]
dates

In [ ]:
# Pandas dataframe 
df = pd.DataFrame({'user_message': messages, 'message_date': dates})

#convert message_date type explicitly into 
df['message_date'] = pd.to_datetime(df['message_date'], format='%d/%m/%y, %I:%M %p - ')  #'%d/%m/%Y, %H:%M - '

df.rename(columns={'message_date': 'date'}, inplace=True)

df.head()

In [ ]:
# How many msg are there
df.shape

In [ ]:
# Pratham Harer: https://youtu.be/Riopy.......
# now separate username and message 

users = []
messages = []

for message in df['user_message']:
    entry = re.split(r'([\w\W]+?):\s', message)   # spliting on basis of :-  Pratham Harer: ---> ([\w\W]+?):\s
    
    if entry[1:]:  # If username exists
        users.append(entry[1])
        messages.append(entry[2])
    else:
        users.append('group_notification')  # the message that doesn't contain a semicolon, like ( Messages to yourself are end-to-end encrypted... )
        messages.append(entry[0])

df['user'] = users
df['message'] = messages

df.drop(columns=['user_message'], inplace=True) # No longer needed coz unstructure so dropped

df.head()

In [ ]:
# now separate date in year, month, day 

In [ ]:
df['year'] = df['date'].dt.year

In [ ]:
df['month'] = df['date'].dt.month_name()

In [ ]:
df['day'] = df['date'].dt.day

In [ ]:
df['hour'] = df['date'].dt.hour

In [ ]:
df['minute'] = df['date'].dt.minute

In [ ]:
df.head()

In [ ]:
########################################################################################################################

In [ ]:
df[df['user'] == 'Pratham Harer'].shape  # total no. of messages

In [ ]:
words = []
for message in df['message']:   # total no. of  words
    words.extend(message.split())

In [ ]:
len(words)

In [ ]:
num_media_messages = df[df['message'] == '<Media omitted>\n'].shape[0] # No. of media shared 
num_media_messages

In [ ]:
!pip install urlextract  # No. of Links shared 

In [ ]:
from urlextract import URLExtract

extractor = URLExtract()

In [ ]:
links = [] 
for message in df['message']:
    links.extend(extractor.find_urls(message))

In [ ]:
links

In [ ]:
len(links)

In [ ]:
# busy user top 5 for group only 
x = df['user'].value_counts().head()

In [ ]:
x

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
name = x.index
count = x.values

In [ ]:
plt.bar(name,count)
plt.xticks(rotation='vertical')
plt.show

In [ ]:
# How many percentage top users did chat among all 

In [ ]:
round((df['user'].value_counts()/df.shape[0])*100,2).reset_index().rename(columns={'index':'name','user':'product'})

In [ ]:
# top 25 most repeated words
# words = []

# for message in df['message']:
    # words.extend(message.split())

In [ ]:
# from collections import Counter 
# pd.DataFrame(Counter(words).most_common(25))      # problem is english words like { is and to for } 

In [ ]:
# remove group notification messages
temp = df[df['user'] != 'group_notification']
# remove media omitted messages
temp = temp[temp['message'] != '<Media omitted>\n']
temp
# remove stop words ( we have to find something that also removes stop words of hindi & marathi ) 

In [ ]:
f = open('stop_hinglish.txt','r')
stop_words = f.read()
print(stop_words)

In [ ]:
# changing above code 
words = []

for message in temp['message']:
    for word in message.lower().split():
        if word not in stop_words:
            words.append(word)

In [ ]:
!pip install emoji 

In [ ]:
emoji_list = []

for message in df['message']:
    emoji_list.extend(emoji.distinct_emoji_list(message))

emoji_df = pd.DataFrame(Counter(emoji_list).most_common(25))

In [ ]:
emoji_df

In [ ]:
# timeline

In [ ]:
df['month_num'] = df['date'].dt.month

In [ ]:
timeline = df.groupby(['year','month_num','month']).count()['message'].reset_index()

In [ ]:
timeline 

In [ ]:
# merge month and year, why so I can plot a bar graph according to message and time ( month - year )
time = []
for i in range(timeline.shape[0]):
    time.append(timeline['month'][i] + "-" + str(timeline['year'][i]))

In [ ]:
time

In [ ]:
timeline['time'] = time

In [ ]:
timeline

In [ ]:
plt.plot(timeline['time'],timeline['message'])
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
# Daily timeline

In [ ]:
df['only_date'] = df['date'].dt.date

In [ ]:
daily_timeline = df.groupby('only_date').count()['message'].reset_index()

In [ ]:
plt.figure(figsize=(18,10))
plt.plot(daily_timeline['only_date'],daily_timeline['message'])

In [ ]:
df.head()

In [ ]:
# Week mey sabse jayda active day konsa hai
df['day_name'] = df['date'].dt.day_name()

In [ ]:
df.head(1)

In [ ]:
df['day_name'].value_counts()

In [ ]:
# Heat map for at what timing user are more active as per day and week

In [ ]:
df.head()

In [ ]:
# hour is in single no. but we need it like this 1-2,2-3,3-4,....

In [ ]:
period = []
period_order = []  # NEW: keep hour as numeric for sorting
for hour in df[['day_name', 'hour']]['hour']:
    hour = int(hour)  # ensure it's an integer

    if hour == 23:
        period.append(f"{hour}-00")  # wrap around for 23
    else:
        period.append(f"{hour}-{hour+1}")  # normal case

    period_order.append(hour)  # store numeric hour

In [ ]:
#new col.
df['period'] = period
df['period_order'] = period_order  # use this for sorting when plotting

In [ ]:
df.sample(5)

In [ ]:
import seaborn as sns 
plt.figure(figsize=(20,6))
sns.heatmap(df.pivot_table(index='day_name', columns='period', values='message', aggfunc='count').fillna(0))
plt.yticks(rotation='horizontal')
plt.show()